# BERTimbau aplicado ao corpus STIL 2023

Este notebook usa o modelo `neuralmind/bert-base-portuguese-cased` para analisar
os artigos presentes em `stil2023_articles (1).json`.

As atividades incluem:

- preparação e divisão dos artigos em blocos;
- embeddings contextuais de artigos e palavras;
- similaridade entre artigos;
- agrupamento e visualização dos documentos;
- previsão de palavras mascaradas;
- pseudo-perplexidade;
- ajuste opcional do BERTimbau ao corpus com *masked language modeling* (MLM);
- comparação entre o modelo original e o modelo ajustado.

> **Importante:** BERTimbau é um modelo de linguagem mascarada. Sua
> pseudo-perplexidade não é diretamente comparável à perplexidade dos modelos
> de bigramas e trigramas, que usam previsão sequencial.


## Etapa 1: instalar dependências

No Google Colab, ative uma GPU em **Ambiente de execução > Alterar tipo de
ambiente de execução > T4 GPU** antes de executar as células.


In [ ]:
!pip -q install -U transformers datasets accelerate scikit-learn seaborn


## Etapa 2: importar bibliotecas e configurar o ambiente

In [ ]:
import json
import math
import random
import re
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from datasets import Dataset
from google.colab import files
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (
    AutoModel,
    AutoModelForMaskedLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    pipeline,
    set_seed,
)

SEED = 42
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
MAX_LENGTH = 512
CHUNK_STRIDE = 64

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)


## Etapa 3: carregar `stil2023_articles (1).json`

Selecione o arquivo quando o botão de upload aparecer. O notebook também aceita
o arquivo já presente em `/content`.


In [ ]:
JSON_NAME = "stil2023_articles (1).json"
json_path = Path("/content") / JSON_NAME

if not json_path.exists():
    uploaded = files.upload()
    if JSON_NAME in uploaded:
        json_path = Path("/content") / JSON_NAME
    else:
        json_path = Path("/content") / next(iter(uploaded))

with json_path.open(encoding="utf-8") as file:
    artigos = json.load(file)

print(f"Arquivo: {json_path.name}")
print(f"Quantidade de artigos: {len(artigos)}")
print("Campos disponíveis:", list(artigos[0]))


## Etapa 4: preparar os textos

In [ ]:
def limpar_texto(texto):
    texto = unicodedata.normalize("NFC", str(texto or ""))
    texto = re.sub(r"https?://\S+|www\.\S+", " ", texto)
    texto = re.sub(r"\S+@\S+", " ", texto)
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()


registros = []
for indice, artigo in enumerate(artigos):
    texto = limpar_texto(
        artigo.get("artigo_completo")
        or artigo.get("artigo_completo_pt")
        or artigo.get("resumo")
    )
    if not texto:
        continue
    registros.append(
        {
            "id": indice,
            "titulo": artigo.get("titulo", f"Artigo {indice + 1}"),
            "idioma": artigo.get("idioma", "Não informado"),
            "texto": texto,
            "caracteres": len(texto),
        }
    )

df_artigos = pd.DataFrame(registros)
display(df_artigos[["id", "titulo", "idioma", "caracteres"]])


## Etapa 5: carregar o BERTimbau e dividir textos longos

O BERTimbau aceita no máximo 512 tokens por entrada. Cada artigo será dividido
em blocos sobrepostos. A sobreposição reduz a perda de contexto nas fronteiras.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
bertimbau = AutoModel.from_pretrained(MODEL_NAME).to(device)
bertimbau.eval()


def dividir_em_blocos(texto, max_length=MAX_LENGTH, stride=CHUNK_STRIDE):
    codificacao = tokenizer(
        texto,
        add_special_tokens=True,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
    )

    blocos = []
    for indice, input_ids in enumerate(codificacao["input_ids"]):
        bloco = {
            "input_ids": input_ids,
            "attention_mask": codificacao["attention_mask"][indice],
        }
        if "token_type_ids" in codificacao:
            bloco["token_type_ids"] = codificacao["token_type_ids"][indice]
        blocos.append(bloco)
    return blocos


df_artigos["quantidade_blocos"] = df_artigos["texto"].apply(
    lambda texto: len(dividir_em_blocos(texto))
)
display(df_artigos[["titulo", "quantidade_blocos"]])


## Etapa 6: gerar embeddings dos artigos

Para cada bloco, é calculada a média dos tokens válidos. O embedding do artigo
é a média dos embeddings de todos os seus blocos.


In [ ]:
def mean_pooling(last_hidden_state, attention_mask):
    mascara = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    soma = torch.sum(last_hidden_state * mascara, dim=1)
    divisor = torch.clamp(mascara.sum(dim=1), min=1e-9)
    return soma / divisor


@torch.inference_mode()
def embedding_artigo(texto):
    vetores = []
    for bloco in dividir_em_blocos(texto):
        input_ids = torch.tensor([bloco["input_ids"]], device=device)
        attention_mask = torch.tensor([bloco["attention_mask"]], device=device)
        saida = bertimbau(input_ids=input_ids, attention_mask=attention_mask)
        vetor = mean_pooling(saida.last_hidden_state, attention_mask)
        vetores.append(vetor.cpu())
    return torch.cat(vetores).mean(dim=0).numpy()


embeddings_artigos = np.vstack(
    [embedding_artigo(texto) for texto in df_artigos["texto"]]
)
np.save("embeddings_artigos_bertimbau.npy", embeddings_artigos)

print("Formato da matriz:", embeddings_artigos.shape)


## Etapa 7: encontrar artigos semanticamente semelhantes

In [ ]:
matriz_similaridade = cosine_similarity(embeddings_artigos)


def artigos_similares(indice, quantidade=5):
    ordem = np.argsort(matriz_similaridade[indice])[::-1]
    ordem = [item for item in ordem if item != indice][:quantidade]
    return pd.DataFrame(
        {
            "artigo_similar": df_artigos.iloc[ordem]["titulo"].values,
            "similaridade": matriz_similaridade[indice, ordem],
        }
    )


INDICE_ARTIGO = 0
print("Artigo consultado:")
print(df_artigos.iloc[INDICE_ARTIGO]["titulo"])
display(artigos_similares(INDICE_ARTIGO))


### Mapa de similaridade entre todos os artigos

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(
    matriz_similaridade,
    cmap="viridis",
    xticklabels=df_artigos["id"],
    yticklabels=df_artigos["id"],
)
plt.title("Similaridade de cosseno entre artigos - BERTimbau")
plt.xlabel("ID do artigo")
plt.ylabel("ID do artigo")
plt.tight_layout()
plt.show()


## Etapa 8: agrupar e visualizar os artigos

In [ ]:
NUM_GRUPOS = 4

kmeans = KMeans(n_clusters=NUM_GRUPOS, random_state=SEED, n_init=20)
df_artigos["grupo"] = kmeans.fit_predict(embeddings_artigos)

pca = PCA(n_components=2, random_state=SEED)
coordenadas = pca.fit_transform(embeddings_artigos)
df_artigos["pca_1"] = coordenadas[:, 0]
df_artigos["pca_2"] = coordenadas[:, 1]

plt.figure(figsize=(11, 7))
sns.scatterplot(
    data=df_artigos,
    x="pca_1",
    y="pca_2",
    hue="grupo",
    palette="tab10",
    s=100,
)
for _, linha in df_artigos.iterrows():
    plt.annotate(
        str(linha["id"]),
        (linha["pca_1"], linha["pca_2"]),
        xytext=(4, 4),
        textcoords="offset points",
    )
plt.title("Artigos STIL 2023 representados por embeddings BERTimbau")
plt.tight_layout()
plt.show()

display(
    df_artigos[["id", "grupo", "titulo"]].sort_values(["grupo", "id"])
)


## Etapa 9: prever palavras mascaradas

Use exatamente um marcador `[MASK]` na frase. O resultado representa as
palavras consideradas mais prováveis pelo BERTimbau naquele contexto.


In [ ]:
preenchedor = pipeline(
    "fill-mask",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device=0 if torch.cuda.is_available() else -1,
)

frase = "O processamento de linguagem natural utiliza modelos de [MASK]."
previsoes = preenchedor(frase, top_k=10)

pd.DataFrame(
    [
        {
            "token": item["token_str"].strip(),
            "probabilidade": item["score"],
            "frase_completa": item["sequence"],
        }
        for item in previsoes
    ]
)


## Etapa 10: comparar uma palavra em diferentes contextos

Esta atividade demonstra a principal diferença entre BERTimbau e Word2Vec:
a representação da palavra muda conforme a frase.


In [ ]:
@torch.inference_mode()
def embedding_palavra(frase, palavra, modelo=bertimbau):
    entradas = tokenizer(
        frase,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    offsets = entradas.pop("offset_mapping")[0].tolist()
    inicio = frase.casefold().find(palavra.casefold())
    if inicio < 0:
        raise ValueError(f"Palavra '{palavra}' não encontrada na frase.")
    fim = inicio + len(palavra)

    indices = [
        indice
        for indice, (token_inicio, token_fim) in enumerate(offsets)
        if token_fim > inicio and token_inicio < fim
    ]
    entradas = {chave: valor.to(device) for chave, valor in entradas.items()}
    saida = modelo(**entradas).last_hidden_state[0, indices]
    return saida.mean(dim=0).cpu().numpy()


palavra = "modelo"
frases = [
    "O modelo de linguagem foi treinado com textos científicos.",
    "O pesquisador apresentou um modelo matemático para o fenômeno.",
    "O novo modelo de carro foi apresentado ao mercado.",
]
vetores = np.vstack([embedding_palavra(frase, palavra) for frase in frases])

display(
    pd.DataFrame(
        cosine_similarity(vetores),
        index=[f"Contexto {i + 1}" for i in range(len(frases))],
        columns=[f"Contexto {i + 1}" for i in range(len(frases))],
    )
)
for indice, frase_contexto in enumerate(frases, start=1):
    print(f"{indice}. {frase_contexto}")


## Etapa 11: calcular pseudo-perplexidade

O algoritmo mascara cada token separadamente e mede a probabilidade atribuída
ao token original. Como o custo cresce com o número de tokens, a demonstração
usa sentenças ou trechos curtos.

> O valor abaixo é uma **pseudo-perplexidade de modelo mascarado**. Não use
> esse número como substituto direto das perplexidades de bigramas e trigramas.


In [ ]:
modelo_mlm = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device)
modelo_mlm.eval()


@torch.inference_mode()
def pseudo_perplexidade(texto, modelo=modelo_mlm, max_tokens=128):
    entradas = tokenizer(
        texto,
        return_tensors="pt",
        truncation=True,
        max_length=max_tokens,
    )
    input_ids = entradas["input_ids"][0]
    attention_mask = entradas["attention_mask"][0]
    especiais = set(tokenizer.all_special_ids)
    posicoes = [
        i for i, token_id in enumerate(input_ids.tolist())
        if token_id not in especiais
    ]
    perdas = []

    for posicao in posicoes:
        ids_mascarados = input_ids.clone()
        token_original = ids_mascarados[posicao].item()
        ids_mascarados[posicao] = tokenizer.mask_token_id

        saida = modelo(
            input_ids=ids_mascarados.unsqueeze(0).to(device),
            attention_mask=attention_mask.unsqueeze(0).to(device),
        )
        log_probs = F.log_softmax(saida.logits[0, posicao], dim=-1)
        perdas.append(-log_probs[token_original].item())

    return math.exp(float(np.mean(perdas))) if perdas else float("nan")


trechos_teste = [
    "A análise do corpus permite identificar padrões recorrentes da escrita científica.",
    "Os modelos de linguagem representam relações entre palavras e contextos.",
]
pd.DataFrame(
    {
        "texto": trechos_teste,
        "pseudo_perplexidade": [
            pseudo_perplexidade(texto) for texto in trechos_teste
        ],
    }
)


## Etapa 12: fine-tuning do BERTimbau no corpus STIL 2023

Esta seção continua o pré-treinamento do BERTimbau com MLM. O corpus contém
apenas 30 artigos, portanto o objetivo é adaptação de domínio, não treinamento
do zero. Use poucas épocas e compare os resultados para observar possível
sobreajuste.

Esta etapa pode levar vários minutos e requer GPU.


In [ ]:
EXECUTAR_AJUSTE = True
CAMINHO_MODELO_AJUSTADO = "/content/bertimbau-stil2023-final"

if EXECUTAR_AJUSTE:
    # Separa por artigo para impedir vazamento de trechos do mesmo documento.
    ids = df_artigos["id"].tolist()
    rng = np.random.default_rng(SEED)
    rng.shuffle(ids)
    n_validacao = max(1, round(len(ids) * 0.2))
    ids_validacao = set(ids[:n_validacao])

    textos_treino, textos_validacao = [], []
    for _, artigo in df_artigos.iterrows():
        destino = (
            textos_validacao
            if artigo["id"] in ids_validacao
            else textos_treino
        )
        for bloco in dividir_em_blocos(artigo["texto"]):
            destino.append(
                tokenizer.decode(
                    bloco["input_ids"],
                    skip_special_tokens=True,
                )
            )

    def criar_dataset(textos):
        dataset = Dataset.from_dict({"text": textos})
        return dataset.map(
            lambda lote: tokenizer(
                lote["text"],
                truncation=True,
                max_length=MAX_LENGTH,
                return_special_tokens_mask=True,
            ),
            batched=True,
            remove_columns=["text"],
        )

    dataset_treino = criar_dataset(textos_treino)
    dataset_validacao = criar_dataset(textos_validacao)
    print("Artigos de treino:", len(ids) - len(ids_validacao))
    print("Artigos de validação:", len(ids_validacao))

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15,
    )
    modelo_ajustado = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

    argumentos = TrainingArguments(
        output_dir="/content/bertimbau-stil2023",
        overwrite_output_dir=True,
        num_train_epochs=2,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        learning_rate=2e-5,
        weight_decay=0.01,
        logging_steps=10,
        save_strategy="epoch",
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED,
    )
    treinador = Trainer(
        model=modelo_ajustado,
        args=argumentos,
        train_dataset=dataset_treino,
        eval_dataset=dataset_validacao,
        data_collator=data_collator,
    )
    treinador.train()
    metricas = treinador.evaluate()
    print(metricas)
    print("Perplexidade MLM aproximada:", math.exp(metricas["eval_loss"]))

    modelo_ajustado.save_pretrained(CAMINHO_MODELO_AJUSTADO)
    tokenizer.save_pretrained(CAMINHO_MODELO_AJUSTADO)
    modelo_contextual_ajustado = AutoModel.from_pretrained(
        CAMINHO_MODELO_AJUSTADO
    ).to(device)
    modelo_contextual_ajustado.eval()


## Etapa 13: comparar o modelo original com o ajustado

In [ ]:
if EXECUTAR_AJUSTE:
    modelo_ajustado = modelo_ajustado.to(device)
    modelo_ajustado.eval()

    comparacao = []
    for texto in trechos_teste:
        comparacao.append(
            {
                "texto": texto,
                "BERTimbau original": pseudo_perplexidade(
                    texto,
                    modelo=modelo_mlm,
                ),
                "BERTimbau ajustado": pseudo_perplexidade(
                    texto,
                    modelo=modelo_ajustado,
                ),
            }
        )
    display(pd.DataFrame(comparacao))
else:
    print("Execute primeiro a etapa de ajuste para gerar esta comparação.")


## Etapa 14: respostas das atividades 1 e 2

O substantivo e o verbo mais frequentes são definidos pelos campos `lema` e
`pos_tagger`. Os vetores são contextuais; portanto, o resultado de cada termo
é a média das suas ocorrências no corpus após o fine-tuning.


In [ ]:
def normalizar_termo(valor):
    valor = unicodedata.normalize("NFC", str(valor or "")).casefold()
    itens = re.findall(r"[a-zà-öø-ÿ]+(?:[-'][a-zà-öø-ÿ]+)?", valor)
    return itens[0] if itens else ""


freq_lemas = Counter()
freq_nomes = Counter()
freq_verbos = Counter()
freq_formas = Counter()

for artigo in artigos:
    tokens = artigo.get("artigo_tokenizado", []) or []
    lemas = artigo.get("lema", []) or []
    tags = artigo.get("pos_tagger", []) or []
    for token, lema, tag in zip(tokens, lemas, tags):
        forma = normalizar_termo(token)
        lema = normalizar_termo(lema) or forma
        if len(forma) <= 2 or len(lema) <= 2:
            continue
        freq_formas[forma] += 1
        freq_lemas[lema] += 1
        if tag == "NOUN":
            freq_nomes[lema] += 1
        elif tag == "VERB":
            freq_verbos[lema] += 1

substantivo_mais_frequente = freq_nomes.most_common(1)[0][0]
verbo_mais_frequente = freq_verbos.most_common(1)[0][0]
termos_atividade = {
    "a_modelos": "modelos",
    "b_linguagem": "linguagem",
    "c_substantivo": substantivo_mais_frequente,
    "d_verbo": verbo_mais_frequente,
}

display(pd.DataFrame([
    {"item": item, "termo": termo}
    for item, termo in termos_atividade.items()
]))


In [ ]:
TERMOS_FORMA_FIXA = {"modelos", "linguagem"}


def blocos_anotados(palavras_por_bloco=300):
    blocos = []
    for artigo in artigos:
        tokens = artigo.get("artigo_tokenizado", []) or []
        lemas = artigo.get("lema", []) or []
        formas, chaves = [], []
        for token, lema in zip(tokens, lemas):
            forma = normalizar_termo(token)
            lema = normalizar_termo(lema) or forma
            if len(forma) <= 2 or len(lema) <= 2:
                continue
            formas.append(forma)
            unidades = [lema]
            if forma in TERMOS_FORMA_FIXA:
                unidades.append(forma)
            chaves.append(tuple(dict.fromkeys(unidades)))
        for inicio in range(0, len(formas), palavras_por_bloco):
            blocos.append((
                formas[inicio:inicio + palavras_por_bloco],
                chaves[inicio:inicio + palavras_por_bloco],
            ))
    return [bloco for bloco in blocos if bloco[0]]


@torch.inference_mode()
def embeddings_contextuais(candidatos, tamanho_lote=8):
    blocos = blocos_anotados()
    somas, contagens = {}, Counter()
    for inicio in range(0, len(blocos), tamanho_lote):
        lote = blocos[inicio:inicio + tamanho_lote]
        entradas = tokenizer(
            [formas for formas, _ in lote],
            is_split_into_words=True,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        estados = modelo_contextual_ajustado(
            **{k: v.to(device) for k, v in entradas.items()}
        ).last_hidden_state.cpu()

        for i, (_, chaves) in enumerate(lote):
            posicoes = defaultdict(list)
            for posicao, palavra_id in enumerate(entradas.word_ids(i)):
                if palavra_id is not None:
                    posicoes[palavra_id].append(posicao)
            for palavra_id, indices in posicoes.items():
                vetor = estados[i, indices].mean(dim=0).numpy()
                for chave in chaves[palavra_id]:
                    if chave not in candidatos:
                        continue
                    somas.setdefault(chave, np.zeros_like(vetor))
                    somas[chave] += vetor
                    contagens[chave] += 1
    return {
        termo: somas[termo] / contagens[termo]
        for termo in somas
    }, contagens


candidatos = {termo for termo, _ in freq_lemas.most_common(500)}
candidatos.update(termos_atividade.values())
embeddings_palavras, contagens_embeddings = embeddings_contextuais(candidatos)

vetores_atividade = pd.DataFrame({
    item: embeddings_palavras[termo]
    for item, termo in termos_atividade.items()
}).T
vetores_atividade.columns = [
    f"dim_{i:03d}" for i in range(vetores_atividade.shape[1])
]
vetores_atividade.insert(
    0, "termo",
    [termos_atividade[item] for item in vetores_atividade.index],
)
display(vetores_atividade.iloc[:, :13])
vetores_atividade.to_csv(
    "ap5_atividade1_vetores.csv",
    encoding="utf-8",
)


In [ ]:
def termos_similares(termo, topn=10):
    palavras = list(embeddings_palavras)
    matriz = np.vstack([embeddings_palavras[p] for p in palavras])
    scores = cosine_similarity(
        embeddings_palavras[termo].reshape(1, -1),
        matriz,
    )[0]
    linhas = []
    for indice in np.argsort(scores)[::-1]:
        similar = palavras[indice]
        if similar == termo:
            continue
        linhas.append({
            "termo_consultado": termo,
            "termo_similar": similar,
            "similaridade": float(scores[indice]),
        })
        if len(linhas) == topn:
            break
    return pd.DataFrame(linhas)


tabelas = []
for item, termo in termos_atividade.items():
    print(f"{item}: termos similares a '{termo}'")
    tabela = termos_similares(termo)
    tabelas.append(tabela)
    display(tabela)

pd.concat(tabelas, ignore_index=True).to_csv(
    "ap5_atividade2_termos_similares.csv",
    index=False,
    encoding="utf-8",
)


## Etapa 15: atividade 3 - classificação do estilo de escrita

A tarefa será uma classificação binária de trechos:

### `Estilo Acadêmico`

- uso de terceira pessoa ou construções impessoais;
- ocorrência de voz passiva e estruturas como `observou-se que`;
- redação objetiva;
- evita primeira pessoa, adjetivos avaliativos, gírias e superlativos
  desnecessários.

### `Estilo Narrativo`

- escrita mais fluida e reflexiva;
- possibilidade de primeira pessoa do plural;
- construções como `analisamos`, `apresentamos` e `propomos`;
- maior aproximação com a forma de ensaio acadêmico.

O JSON não possui esses rótulos. A célula seguinte cria uma **sugestão
heurística**, mas a coluna `rotulo_final` deve ser revisada manualmente. O
BERTimbau classificador só deve ser treinado depois dessa anotação.


In [ ]:
PADROES_ACADEMICOS = [
    r"\b(?:observou|verificou|constatou|identificou)-se\b",
    r"\b(?:foi|foram|é|são)\s+\w+(?:ado|ada|idos|idas)\b",
    r"\b(?:este|o presente) trabalho\b",
]
PADROES_NARRATIVOS = [
    r"\b(?:nós|nosso|nossa|nossos|nossas)\b",
    r"\b(?:analisamos|investigamos|apresentamos|propomos|observamos)\b",
]


def segmentar_texto(texto, minimo=250, maximo=900):
    sentencas = re.split(r"(?<=[.!?])\s+", limpar_texto(texto))
    segmentos, atual, tamanho = [], [], 0
    for sentenca in sentencas:
        atual.append(sentenca)
        tamanho += len(sentenca)
        if tamanho >= minimo:
            segmentos.append(" ".join(atual)[:maximo])
            atual, tamanho = [], 0
    return segmentos


def sugerir_estilo(texto):
    texto = texto.casefold()
    academico = sum(
        len(re.findall(padrao, texto))
        for padrao in PADROES_ACADEMICOS
    )
    narrativo = sum(
        len(re.findall(padrao, texto))
        for padrao in PADROES_NARRATIVOS
    )
    if narrativo > academico:
        sugestao = "Estilo Narrativo"
    elif academico > narrativo:
        sugestao = "Estilo Acadêmico"
    else:
        sugestao = "Revisão manual"
    return sugestao, academico, narrativo


amostras = []
for _, artigo in df_artigos.iterrows():
    for numero, trecho in enumerate(
        segmentar_texto(artigo["texto"])[:20],
        start=1,
    ):
        sugestao, sinais_academicos, sinais_narrativos = sugerir_estilo(trecho)
        amostras.append({
            "artigo_id": artigo["id"],
            "segmento_id": numero,
            "titulo": artigo["titulo"],
            "texto": trecho,
            "sugestao_heuristica": sugestao,
            "sinais_academicos": sinais_academicos,
            "sinais_narrativos": sinais_narrativos,
            "rotulo_final": "",
        })

df_estilos = pd.DataFrame(amostras)
df_estilos.to_csv(
    "ap5_amostras_classificacao_estilo.csv",
    index=False,
    encoding="utf-8",
)
display(df_estilos.head(10))
print("Preencha rotulo_final com Estilo Acadêmico ou Estilo Narrativo.")


### Treinamento e avaliação propostos

Após a revisão dos rótulos:

1. usar `BertForSequenceClassification` com duas classes;
2. separar treino, validação e teste por artigo;
3. nunca dividir aleatoriamente apenas os trechos;
4. avaliar macro-F1, F1 de cada classe e matriz de confusão;
5. comparar com TF-IDF + regressão logística;
6. verificar equilíbrio entre as classes e concordância entre anotadores.

As expressões linguísticas ajudam a definir os rótulos, mas o classificador
deve aprender com os trechos completos, não apenas procurar palavras-chave.


## Etapa 16: exportar resultados

São exportados os metadados dos grupos, a matriz de similaridade e os
embeddings dos artigos.


In [ ]:
df_artigos.drop(columns=["texto"]).to_csv(
    "artigos_bertimbau_grupos.csv",
    index=False,
    encoding="utf-8",
)
pd.DataFrame(
    matriz_similaridade,
    index=df_artigos["id"],
    columns=df_artigos["id"],
).to_csv("similaridade_artigos_bertimbau.csv", encoding="utf-8")
np.save("embeddings_artigos_bertimbau.npy", embeddings_artigos)

print("Arquivos gerados:")
print("- artigos_bertimbau_grupos.csv")
print("- similaridade_artigos_bertimbau.csv")
print("- embeddings_artigos_bertimbau.npy")


## Interpretação dos resultados

- **Similaridade:** valores próximos de 1 indicam representações semelhantes,
  mas não provam que os artigos têm o mesmo tema ou qualidade.
- **Agrupamento:** os grupos do K-Means são exploratórios e dependem do número
  de grupos escolhido.
- **PCA:** a projeção em duas dimensões perde parte da informação dos vetores.
- **Palavras contextuais:** a mesma palavra pode receber vetores diferentes em
  frases diferentes.
- **Pseudo-perplexidade:** deve ser usada para comparar versões compatíveis do
  BERTimbau, não para declarar superioridade sobre modelos de n-gramas.
- **Ajuste no corpus:** redução da perda no próprio corpus pode representar
  adaptação ao domínio ou sobreajuste. Uma avaliação externa seria necessária
  para uma conclusão geral.
